In [1]:
from nnsight import LanguageModel
model = LanguageModel("meta-llama/Llama-3.1-8B", device_map="cuda", dispatch=True)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [2]:
import json

with open("neurons_per_task.json", 'r') as f:
    addition_neuron_set = json.load(f)["addition"]
print(len(addition_neuron_set))

# Number of random-control resamples per eval. For each eval, we draw
# N_RANDOM_SAMPLES disjoint-from-addition layer-18 subsets of the same size
# as addition_neuron_set, zero-ablate each, and report the mean (and std)
# of accuracy across the resamples.
N_RANDOM_SAMPLES = 10

28


In [3]:
import torch
import random as _random
from tasks import TASKS
from utils import load_task_info

def eval_task_ablated(task, cycles, template=None, inp_map=None, num_map=None,
                      n_random_samples=None):
    """Evaluate clean/ablated accuracy on a cyclic task with an optional rephrased template.

    Template placeholders: `{input}` for the starting state, `{number}` or `{offset}` for the offset.
    `inp_map` / `num_map` map what appears in the prompt (key) -> what the causal model receives (value).

    Random control is resampled `n_random_samples` times (defaults to N_RANDOM_SAMPLES from the
    enclosing notebook); each resample is a disjoint-from-addition subset of layer-18 neurons of
    the same size as addition_neuron_set. Returns mean and std of the random-control accuracy.
    """
    if n_random_samples is None:
        n_random_samples = N_RANDOM_SAMPLES

    cm = TASKS[task]["causal_model"]
    num_to_idx = TASKS[task]["num_to_idx"]

    # preserve (possibly inverted) maps so the result row reflects what's in the prompt
    orig_inp_map = inp_map
    orig_num_map = num_map

    default_inps, default_nums, default_template = load_task_info(task)
    if template is None:
        template = default_template

    if inp_map is None:
        inps = list(default_inps)
        inp_map = {i : i for i in inps}
    else:
        inps = list(inp_map.keys())
    inp_map_r = {v : k for k, v in inp_map.items()}

    if num_map is None:
        nums = list(default_nums)
        num_map = {n : n for n in nums}
    else:
        nums = list(num_map.keys())

    prompt_dict = {}
    for num in nums:
        cm_num = num_map[num]
        check_num = cm_num
        if num_to_idx(check_num) <= cycles * len(inps):
            for inp in inps:
                # accept both {number} and {offset} in templates
                prompt_dict[(inp, num)] = template.format(number=num, offset=num, input=inp)

    answers = []
    for tup, p in prompt_dict.items():
        cm_inp = inp_map[tup[0]]
        cm_num = num_map[tup[1]]

        # causal models use "offset" as the variable name (not "number")
        ans = cm.run_forward({"input" : cm_inp, "offset" : cm_num})["raw_output"]
        if task == "hours":
            ans = ans.split(':')[0].strip()
            answers.append(inp_map_r[ans])
        elif task == "addition":
            answers.append(ans.strip())
        else:
            answers.append(" " + inp_map_r[ans.strip()])

    # For addition: leading space on the predicted token is inconsistent, and the model
    # may emit either the digit ("30") or the word form ("thirty"). Strip and normalize
    # both sides to digit form before comparing.
    def match(p, a):
        if task == "addition":
            return p.strip() == a.strip()
        return p in a or a in p # to include Mon = Monday

    prompts = list(prompt_dict.values())

    # Clean
    with torch.no_grad():
        with model.trace(prompts):
            clean_toks = model.output.logits[:, -1].argmax(dim=-1).save()

    # Zero-ablate addition neurons
    with torch.no_grad():
        with model.trace(prompts):
            for neuron_idx in addition_neuron_set:
                model.model.layers[18].mlp.down_proj.input[:, -1, neuron_idx] = 0
            zeroed_toks = model.output.logits[:, -1].argmax(dim=-1).save()

    # Flip addition neurons
    with torch.no_grad():
        with model.trace(prompts):
            for neuron_idx in addition_neuron_set:
                model.model.layers[18].mlp.down_proj.input[:, -1, neuron_idx] *= -1
            flipped_toks = model.output.logits[:, -1].argmax(dim=-1).save()

    # Keep only addition neurons (zero everything else)
    with torch.no_grad():
        with model.trace(prompts):
            for neuron_idx in range(model.config.intermediate_size):
                if neuron_idx not in addition_neuron_set:
                    model.model.layers[18].mlp.down_proj.input[:, -1, neuron_idx] = 0
            only_toks = model.output.logits[:, -1].argmax(dim=-1).save()

    clean_preds = [model.tokenizer.decode(t) for t in clean_toks]
    zeroed_preds = [model.tokenizer.decode(t) for t in zeroed_toks]
    flipped_preds = [model.tokenizer.decode(t) for t in flipped_toks]
    only_preds = [model.tokenizer.decode(t) for t in only_toks]

    clean_acc = sum(match(p, a) for p, a in zip(clean_preds, answers)) / len(answers)
    zeroed_acc = sum(match(p, a) for p, a in zip(zeroed_preds, answers)) / len(answers)
    flipped_acc = sum(match(p, a) for p, a in zip(flipped_preds, answers)) / len(answers)
    only_acc = sum(match(p, a) for p, a in zip(only_preds, answers)) / len(answers)

    # Random control: resample n_random_samples disjoint subsets and average.
    candidates = [i for i in range(model.config.intermediate_size) if i not in addition_neuron_set]
    random_accs = []
    for sample_idx in range(n_random_samples):
        rng = _random.Random(1000 + sample_idx)
        random_neuron_subset = rng.sample(candidates, len(addition_neuron_set))
        with torch.no_grad():
            with model.trace(prompts):
                for neuron_idx in random_neuron_subset:
                    model.model.layers[18].mlp.down_proj.input[:, -1, neuron_idx] = 0
                random_probs = model.output.logits[:, -1].softmax(dim=-1).save()
        random_preds = [model.tokenizer.decode(t) for t in random_probs.argmax(dim=-1)]
        random_accs.append(sum(match(p, a) for p, a in zip(random_preds, answers)) / len(answers))

    random_mean = sum(random_accs) / len(random_accs)
    _var = sum((x - random_mean) ** 2 for x in random_accs) / len(random_accs)
    random_std = _var ** 0.5

    if (clean_acc < 0.8):
        for pmpt, pred, answer in zip(list(prompt_dict.values()), clean_preds, answers):
            if not match(pred, answer):
                print(repr(pmpt), repr(pred), repr(answer))

    if (zeroed_acc > 0.8):
        for pmpt, pred, answer in zip(list(prompt_dict.values()), clean_preds, answers):
            if match(pred, answer):
                print(repr(pmpt), repr(pred), repr(answer))

    print(f"{task.upper()} ({cycles} cycle) | template: {template[:60]}...")
    print(f"  Clean:              {clean_acc:.2%}")
    print(f"  Neurons zeroed:     {zeroed_acc:.2%}")
    print(f"  Neurons flipped:    {flipped_acc:.2%}")
    print(f"  Only these neurons: {only_acc:.2%}")
    print(f"  Random zeroed:      {random_mean:.2%} ± {random_std:.2%}  (n={n_random_samples})")
    print(f"    per-sample: " + ", ".join(f"{a:.2%}" for a in random_accs))
    print()

    return {
        "task": task,
        "template": template,
        "inp_map": orig_inp_map,
        "num_map": orig_num_map,
        "clean": clean_acc,
        "zeroed": zeroed_acc,
        "flipped": flipped_acc,
        "only": only_acc,
        "random_mean": random_mean,
        "random_std": random_std,
        "random_samples": random_accs,
        "n_random_samples": n_random_samples,
    }

In [5]:
_TABLE_TEMPLATE = r"""\begin{table}[h!]
\caption{Neuron intervention results for the \texttt{%(task)s} task across alternative prompt templates, for one cycle each. \textbf{Input} / \textbf{Offset Format} columns show examples of strings input to model. \textbf{Clean}: unmodified accuracy for \llama. \textbf{Only}: accuracy when all neurons at layer 18 except for $\mathcal{N}_{\text{add}}$ are zero-ablated. \textbf{Zeroed}: accuracy when all %(n)d $\mathcal{N}_{\text{add}}$ addition neurons are zero-ablated. \textbf{Flipped}: accuracy when %(n)d $\mathcal{N}_{\text{add}}$ neurons have their signs flipped. \textbf{Random}: accuracy when %(n)d random non-addition neurons at layer 18 are zero-ablated, averaged (mean $\pm$ std) over %(n_random)d resamples.}
\begin{tabular}{@{}>{\raggedright\arraybackslash}p{2.2cm}>{\raggedright\arraybackslash}p{1.5cm}>{\raggedright\arraybackslash}p{1.5cm}>{\raggedright\arraybackslash}p{1.05cm}>{\raggedright\arraybackslash}p{1.05cm}>{\raggedright\arraybackslash}p{1.05cm}>{\raggedright\arraybackslash}p{1.05cm}>{\raggedright\arraybackslash}p{2.3cm}l@{}}
\toprule
\bf Template & \bf Input Format & \bf Offset Format & \bf Clean & \bf Only $\uparrow$ & \bf Zeroed $\downarrow$ & \bf Flipped $\downarrow$ & \bf Random $\uparrow$ \\ \midrule
%(rows)s
\bottomrule
\end{tabular}
\label{tab:rephrased-%(task)s}
\end{table}"""


def _escape_template(t):
    return t.replace("{", r"\{").replace("}", r"\}").replace("%", r"\%")


def _pct(x):
    return f"{100 * x:.2f}\\%"


def _pct_mean_std(m, s):
    return f"{100 * m:.2f}\\% $\\pm$ {100 * s:.2f}\\%"


def _format_keys(m, defaults, max_items=2):
    """Show the prompt-side strings: map.keys() if a map was passed, otherwise `defaults`.
    Truncated to `max_items` with '...' when there are more."""
    items = list(m.keys()) if m is not None else list(defaults)
    if len(items) > max_items:
        return ", ".join(items[:max_items]) + "..."
    return ", ".join(items)


def format_task_latex(task, results, n_neurons=None):
    """Format per-task results as a LaTeX table. Default-prompt rows show '(default prompt)'.

    The Random column reports mean $\\pm$ std over n_random_samples resampled subsets.
    """
    default_inps, default_nums, default_template = load_task_info(task)
    if n_neurons is None:
        n_neurons = len(addition_neuron_set)

    n_random = results[0].get("n_random_samples", N_RANDOM_SAMPLES) if results else N_RANDOM_SAMPLES

    rows = []
    for r in results:
        if r["template"] == default_template:
            cell = r"\it (default prompt)"
        else:
            cell = _escape_template(r["template"])
        rows.append(
            f"{cell} & {_format_keys(r['inp_map'], default_inps)} & {_format_keys(r['num_map'], default_nums)} "
            f"& {_pct(r['clean'])} & {_pct(r['only'])} & {_pct(r['zeroed'])} & {_pct(r['flipped'])} "
            f"& {_pct_mean_std(r['random_mean'], r['random_std'])} \\\\"
        )

    return _TABLE_TEMPLATE % {
        "task": task,
        "n": n_neurons,
        "n_random": n_random,
        "rows": "\n".join(rows),
    }

In [6]:
# Reusable maps shared across tasks.
# Each map's keys are what appears in the prompt; values are what the causal model receives.

my_num_map = {
    "1": "one",
    "2": "two",
    "3": "three",
    "4": "four",
    "5": "five",
    "6": "six",
    "7": "seven",
    "8": "eight",
    "9": "nine",
    "10": "ten",
    "11": "eleven",
    "12": "twelve",
    "13": "thirteen",
    "14": "fourteen",
    "15": "fifteen",
    "16": "sixteen",
    "17": "seventeen",
    "18": "eighteen",
    "19": "nineteen",
    "20": "twenty",
    "21": "twenty-one",
    "22": "twenty-two",
    "23": "twenty-three",
    "24": "twenty-four",
}


## Weekdays

In [7]:
weekday_inp_map = {
    "Mon": "Monday",
    "Tues": "Tuesday",
    "Wed": "Wednesday",
    "Thu": "Thursday",
    "Fri": "Friday",
    "Sat": "Saturday",
    "Sun": "Sunday",
}

weekday_num_map = {k: v for k, v in my_num_map.items() if int(k) <= len(weekday_inp_map)}

In [8]:
weekday_experiments = [
    # default 
    (1, None, None, None),

    # same writing but diff template 
    (1, "Let's do some weekday math. {number} days after {input} is what day? Answer:", None, None),

    # diff numbers
    (1, "Question: Today is {input}. In {number} days, what day will it be? Answer: It will be", None, weekday_num_map),

    # diff weekdays 
    (1, "If today is {input}, and I have a deadline in {number} days, what day is my deadline? My deadline is on", weekday_inp_map, None),

    # diff numbers and weekdays
    (1, "Let's do some calendar math. Today is {input}. What day will it be in {number} days? Answer:", weekday_inp_map, weekday_num_map),
]

weekday_results = []
for cycles, template, inp_map, num_map in weekday_experiments:
    res = eval_task_ablated("weekdays", cycles, template=template, inp_map=inp_map, num_map=num_map)
    weekday_results.append(res)

WEEKDAYS (1 cycle) | template: Q: What day is {offset} days after {input}?
A:...
  Clean:              91.84%
  Neurons zeroed:     40.82%
  Neurons flipped:    20.41%
  Only these neurons: 89.80%
  Random zeroed:      92.45% ± 1.31%  (n=10)
    per-sample: 91.84%, 93.88%, 93.88%, 91.84%, 91.84%, 89.80%, 91.84%, 93.88%, 93.88%, 91.84%

"Let's do some weekday math. four days after Friday is what day? Answer:" ' Monday' ' Tuesday'
"Let's do some weekday math. five days after Monday is what day? Answer:" ' Friday' ' Saturday'
"Let's do some weekday math. five days after Friday is what day? Answer:" ' Tuesday' ' Wednesday'
"Let's do some weekday math. five days after Saturday is what day? Answer:" ' Wednesday' ' Thursday'
"Let's do some weekday math. six days after Tuesday is what day? Answer:" ' Sunday' ' Monday'
"Let's do some weekday math. six days after Wednesday is what day? Answer:" ' Monday' ' Tuesday'
"Let's do some weekday math. six days after Friday is what day? Answer:" ' Monday

In [9]:
print(format_task_latex("weekdays", weekday_results))

\begin{table}[h!]
\caption{Neuron intervention results for the \texttt{weekdays} task across alternative prompt templates, for one cycle each. \textbf{Input} / \textbf{Offset Format} columns show examples of strings input to model. \textbf{Clean}: unmodified accuracy for \llama. \textbf{Only}: accuracy when all neurons at layer 18 except for $\mathcal{N}_{\text{add}}$ are zero-ablated. \textbf{Zeroed}: accuracy when all 28 $\mathcal{N}_{\text{add}}$ addition neurons are zero-ablated. \textbf{Flipped}: accuracy when 28 $\mathcal{N}_{\text{add}}$ neurons have their signs flipped. \textbf{Random}: accuracy when 28 random non-addition neurons at layer 18 are zero-ablated, averaged (mean $\pm$ std) over 10 resamples.}
\begin{tabular}{@{}>{\raggedright\arraybackslash}p{2.2cm}>{\raggedright\arraybackslash}p{1.5cm}>{\raggedright\arraybackslash}p{1.5cm}>{\raggedright\arraybackslash}p{1.05cm}>{\raggedright\arraybackslash}p{1.05cm}>{\raggedright\arraybackslash}p{1.05cm}>{\raggedright\arraybacksla

## Months


In [10]:
months_inp_map = {
    "Jan": "January",
    "Feb": "February",
    "Mar": "March",
    "Apr": "April",
    "May": "May",
    "Jun": "June",
    "Jul": "July",
    "Aug": "August",
    "Sep": "September",
    "Oct": "October",
    "Nov": "November",
    "Dec": "December",
}
months_num_map = {k: v for k, v in my_num_map.items() if int(k) <= len(months_inp_map)}

In [11]:
months_experiments = [
    # default template
    (1, None, None, None),

    # same writing style but diff template 
    (1, "Let's do some calendar math. If the current month is {input}, then in {number} months, what month will it be? Answer: It will be", None, None),

    # diff numbers
    (1, "Question: I have a big presentation in {number} months. If it is currently {input}, then what month is my presentation? Answer: My presentation is in", None, months_num_map),

    # diff months
    (1, "Q: A pregnant woman will give birth in {number} months. If it is currently {input}, what month will she give birth? Answer: She will give birth in", months_inp_map, None),

    # dif numbers and months
    (1, "Let's figure out some scheduling. I just got my hair cut, and it is currently {input}. If I have to book another appointment in {number} months, what month will it be? Answer: My appointment is in", months_inp_map, months_num_map),
]


months_results = []
for cycles, template, inp_map, num_map in months_experiments:
    res = eval_task_ablated("months", cycles, template=template, inp_map=inp_map, num_map=num_map)
    months_results.append(res)

MONTHS (1 cycle) | template: Q: What month is {offset} months after {input}?
A:...
  Clean:              81.25%
  Neurons zeroed:     36.81%
  Neurons flipped:    14.58%
  Only these neurons: 76.39%
  Random zeroed:      81.32% ± 1.00%  (n=10)
    per-sample: 81.25%, 81.94%, 81.94%, 81.25%, 81.25%, 78.47%, 81.94%, 81.94%, 81.25%, 81.94%

MONTHS (1 cycle) | template: Let's do some calendar math. If the current month is {input}...
  Clean:              97.22%
  Neurons zeroed:     59.03%
  Neurons flipped:    22.22%
  Only these neurons: 96.53%
  Random zeroed:      96.74% ± 0.82%  (n=10)
    per-sample: 97.22%, 96.53%, 96.53%, 96.53%, 97.22%, 94.44%, 97.22%, 97.22%, 97.22%, 97.22%

'Question: I have a big presentation in 1 months. If it is currently January, then what month is my presentation? Answer: My presentation is in' ' March' ' February'
'Question: I have a big presentation in 1 months. If it is currently February, then what month is my presentation? Answer: My presentation is in

In [12]:
print(format_task_latex("months", months_results))

\begin{table}[h!]
\caption{Neuron intervention results for the \texttt{months} task across alternative prompt templates, for one cycle each. \textbf{Input} / \textbf{Offset Format} columns show examples of strings input to model. \textbf{Clean}: unmodified accuracy for \llama. \textbf{Only}: accuracy when all neurons at layer 18 except for $\mathcal{N}_{\text{add}}$ are zero-ablated. \textbf{Zeroed}: accuracy when all 28 $\mathcal{N}_{\text{add}}$ addition neurons are zero-ablated. \textbf{Flipped}: accuracy when 28 $\mathcal{N}_{\text{add}}$ neurons have their signs flipped. \textbf{Random}: accuracy when 28 random non-addition neurons at layer 18 are zero-ablated, averaged (mean $\pm$ std) over 10 resamples.}
\begin{tabular}{@{}>{\raggedright\arraybackslash}p{2.2cm}>{\raggedright\arraybackslash}p{1.5cm}>{\raggedright\arraybackslash}p{1.5cm}>{\raggedright\arraybackslash}p{1.05cm}>{\raggedright\arraybackslash}p{1.05cm}>{\raggedright\arraybackslash}p{1.05cm}>{\raggedright\arraybackslash

## Hours

In [13]:
hours_num_map = {k: v for k, v in my_num_map.items() if int(k) <= 24}

In [14]:
hours_experiments = [
    # default template
    (1, None, None, None),

    # same writing style but diff template 
    (1, "Let's do some clock math. If it is {input}:00 right now in 24-hour time, then what time will it be in {number} hours? Answer: ", None, None),

    # same writing style but diff template 
    (1, "When I was in the military, I learned 24-hour time. So if right now it is {input}:00, then in {number} hours, it will be ", None, None),

    # same writing style but diff template 
    (1, "Q: If it is {input} o' clock in military time, what time will it be in {number} hours? A: In 24-hour time, it will be ", None, None),

    # different offsets
    (1, "I need to work out when to get to the airport. Q: It's {input}:00 right now in 24-hour time. My flight is in {number} hours. What time is my flight? Answer: My flight will be at ", None, hours_num_map),
]

hours_results = []
for cycles, template, inp_map, num_map in hours_experiments:
    res = eval_task_ablated("hours", cycles, template=template, inp_map=inp_map, num_map=num_map)
    hours_results.append(res)

HOURS (1 cycle) | template: Q: In 24-hour time, it is now {input}:00. What time will it ...
  Clean:              97.57%
  Neurons zeroed:     50.87%
  Neurons flipped:    16.15%
  Only these neurons: 84.55%
  Random zeroed:      97.24% ± 0.28%  (n=10)
    per-sample: 97.22%, 97.22%, 97.22%, 97.40%, 97.22%, 96.53%, 97.57%, 97.40%, 97.57%, 97.05%

HOURS (1 cycle) | template: Let's do some clock math. If it is {input}:00 right now in 2...
  Clean:              90.45%
  Neurons zeroed:     47.40%
  Neurons flipped:    13.37%
  Only these neurons: 68.40%
  Random zeroed:      90.31% ± 1.30%  (n=10)
    per-sample: 90.62%, 91.15%, 90.62%, 90.97%, 90.80%, 86.46%, 90.45%, 90.45%, 90.62%, 90.97%

HOURS (1 cycle) | template: When I was in the military, I learned 24-hour time. So if ri...
  Clean:              91.32%
  Neurons zeroed:     50.52%
  Neurons flipped:    18.06%
  Only these neurons: 83.68%
  Random zeroed:      91.15% ± 0.28%  (n=10)
    per-sample: 91.32%, 90.97%, 91.32%, 91.49%, 9

In [15]:
print(format_task_latex("hours", hours_results))

\begin{table}[h!]
\caption{Neuron intervention results for the \texttt{hours} task across alternative prompt templates, for one cycle each. \textbf{Input} / \textbf{Offset Format} columns show examples of strings input to model. \textbf{Clean}: unmodified accuracy for \llama. \textbf{Only}: accuracy when all neurons at layer 18 except for $\mathcal{N}_{\text{add}}$ are zero-ablated. \textbf{Zeroed}: accuracy when all 28 $\mathcal{N}_{\text{add}}$ addition neurons are zero-ablated. \textbf{Flipped}: accuracy when 28 $\mathcal{N}_{\text{add}}$ neurons have their signs flipped. \textbf{Random}: accuracy when 28 random non-addition neurons at layer 18 are zero-ablated, averaged (mean $\pm$ std) over 10 resamples.}
\begin{tabular}{@{}>{\raggedright\arraybackslash}p{2.2cm}>{\raggedright\arraybackslash}p{1.5cm}>{\raggedright\arraybackslash}p{1.5cm}>{\raggedright\arraybackslash}p{1.05cm}>{\raggedright\arraybackslash}p{1.05cm}>{\raggedright\arraybackslash}p{1.05cm}>{\raggedright\arraybackslash}

## Addition

In [5]:
addition_experiments = [
    # (1, None, None, None),

    # (1, "The sum of {input} and {number} equals ", None, None),
 
    # (1, "Math question: {input} plus {number} equals ", None, None),

    # (1, "Q: What is the sum of {input} and {number}? A: ", None, None),

    (1, "Q: If I have {input} apples and buy {number} more, how many apples do I have? Total apples: ", None, None),
]

addition_results = []
for cycles, template, inp_map, num_map in addition_experiments:
    res = eval_task_ablated("addition", cycles, template=template, inp_map=inp_map, num_map=num_map)
    addition_results.append(res)

'Q: If I have 1 apples and buy 1 more, how many apples do I have? Total apples: ' '1' '2'
'Q: If I have 3 apples and buy 1 more, how many apples do I have? Total apples: ' '3' '4'
'Q: If I have 5 apples and buy 1 more, how many apples do I have? Total apples: ' '5' '6'
'Q: If I have 6 apples and buy 1 more, how many apples do I have? Total apples: ' '6' '7'
'Q: If I have 7 apples and buy 1 more, how many apples do I have? Total apples: ' '7' '8'
'Q: If I have 25 apples and buy 1 more, how many apples do I have? Total apples: ' '25' '26'
'Q: If I have 27 apples and buy 1 more, how many apples do I have? Total apples: ' '27' '28'
'Q: If I have 30 apples and buy 1 more, how many apples do I have? Total apples: ' '30' '31'
'Q: If I have 33 apples and buy 1 more, how many apples do I have? Total apples: ' '33' '34'
'Q: If I have 35 apples and buy 1 more, how many apples do I have? Total apples: ' '35' '36'
'Q: If I have 37 apples and buy 1 more, how many apples do I have? Total apples: ' '3

In [ ]:
print(format_task_latex("addition", addition_results))